# Simplified Transform Analysis
## Target star transformation

Now let's see how spectral chrarcteristics of the measurement channel
affects transform values.

In [ ]:
from vsopy.util import ValErr
from measurement import SimpleMeasurement
from experiment import Experiment
from sequence import StarSequence
import numpy as np
from astropy.table import QTable

SEED = 42


EXPERIMENT = Experiment(SEED)

SEQUENCE = EXPERIMENT.generate_sequence(
    Experiment.GenParams(
        num_stars=20,
        mag=(10.0, 15.0),
        ci=(-0.5, 1.0),
        uncert=ValErr(0.05, 0.05)))
SEQUENCE.stars

In [ ]:
TARGET = EXPERIMENT.create_target([10, 11], [.01, .01], SEQUENCE.bands)
TARGET.stars

In [ ]:
from measurement import SkewedMeasurement
from vsopy.phot import apply_simple_transform, create_simple_transform
from vsopy.util import MagErrDtype

TARGET_MAG = np.linspace(8.0, 17.0, 10)
TARGET_CI = np.linspace(-0.8, 1.8, 9)
MEASUREMENT = EXPERIMENT.create_measurement(
                SkewedMeasurement,
                params = SimpleMeasurement.Params(
                    offset=1.0,
                    scatter=0.05,
                    uncert=ValErr(0.03, 0.03)),
                skew=(-0.1, 0.1))

def transform_target(target, seq, num_measurements):
    measured_target = [MEASUREMENT(target) for _ in range(num_measurements)]
    measured_ensemble = [EXPERIMENT.measure_ensemble(seq, MEASUREMENT) for _ in range(num_measurements)]
    bands = SEQUENCE.bands
    std_A = SEQUENCE.ensemble[seq.ensemble['band'] == bands[0]]['M']['mag']
    std_B = SEQUENCE.ensemble[seq.ensemble['band'] == bands[1]]['M']['mag']
    std_comp = SEQUENCE.comp_star
    result = [apply_simple_transform(
        create_simple_transform(std_A, std_B,
                                ensemble[ensemble['band'] == bands[0]]['instr M']['mag'],
                                ensemble[ensemble['band'] == bands[1]]['instr M']['mag']),
        std_comp[std_comp['band'] == bands[0]]['M'][0],
        std_comp[std_comp['band'] == bands[1]]['M'][0],
        comp[comp['band'] == bands[0]]['instr M'][0],
        comp[comp['band'] == bands[1]]['instr M'][0],
        target[target['band'] == bands[0]]['instr M'][0],
        target[target['band'] == bands[1]]['instr M'][0],
    ) for (ensemble, comp), target in zip(measured_ensemble, measured_target)]
    return QTable([{'A': row[0], 'B': row[3], 'A1': row[2], 'B1': row[1]} for row in result],
                  dtype=[MagErrDtype] * 4)

transform_target(TARGET, SEQUENCE, 3)

In [ ]:
def target_stats(tv, std):
    return {
        'A': np.mean(tv['A']['mag'] - std[0]),
        'A.std': np.std(tv['A']['mag']),
        'A.err': np.mean(tv['A']['err']),
        'B': np.mean(tv['B']['mag'] - std[1]),
        'B.std': np.std(tv['B']['mag']),
        'B.err': np.mean(tv['B']['err'])
    }

TG_STATS = QTable([target_stats(
    transform_target(
        EXPERIMENT.create_target(
            (mag_A, mag_A+ci),
            (.01, .01),
            SEQUENCE.bands),
        SEQUENCE,
        100),
    (mag_A, mag_A+ci)) | {'std A': mag_A, 'std CI': ci}
 for mag_A in TARGET_MAG for ci in TARGET_CI])

TG_STATS

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

X = np.array(TG_STATS['std A']).reshape(len(TARGET_MAG), len(TARGET_CI))
Y = np.array(TG_STATS['std CI']).reshape(len(TARGET_MAG), len(TARGET_CI))

PREFIX = ['A', 'B']
SUFFIX = ['', '.std', '.err']
FIG, AXS = plt.subplots(len(PREFIX), len(SUFFIX), figsize=(10.24, 10.24), constrained_layout=True)


def plot_contour(ax, column):
    z = np.array(TG_STATS[column]).reshape(len(TARGET_MAG), len(TARGET_CI))
    #ax = FIG.add_subplot(GS[row, col])
    #ax.set_box_aspect(1)
    cont = ax.contourf(X, Y, z, levels=20, cmap='viridis')
    cb = FIG.colorbar(cont, ax=ax, fraction=0.046, pad=0.04, orientation="horizontal")
    cb.locator = mpl.ticker.MaxNLocator(nbins=5)
    cb.update_ticks()
    if row == len(PREFIX) - 1:
        ax.set_xlabel('A')
    if col == 0:
        ax.set_ylabel('CI')
    ax.set_title(column)
    return cont

for col in range(len(SUFFIX)):
    for row in range(len(PREFIX)):
        plot_contour(AXS[row, col], f'{PREFIX[row]}{SUFFIX[col]}')

plt.show()